# 의료AI 해커톤 — 자동 실행 노트북

**사용법**: 런타임 → 모두 실행 (Ctrl+F9). 약 50~55분 후 모든 결과 자동 출력.

## 안전장치
- GPU 런타임 아니면 즉시 중단
- 모든 체크포인트는 Google Drive에 자동 저장 (세션 끊겨도 안전)
- `!python -u` 직접 호출로 출력 실시간 streaming
- axial부터 모두 새로 학습 (이전 부분 학습 흔적 정리)

## 단계 (순차 자동 실행)
1. 환경 검증 (GPU + PyTorch + Drive)
2. Drive 마운트
3. 안전 클론 + symlink
4. 패키지 설치
5. 파이프라인 (View A → C → Merge → Few-shot, 약 50분)
6. Merge grid 결과 표

**시작 전 권장**: 브라우저 콘솔(F12)에 keep-alive 등록 (50분 자리 비우려면)

```javascript
function clickConnect() {
    console.log("keep-alive " + new Date().toLocaleTimeString());
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(clickConnect, 60000);
```

## 1. 환경 검증 — GPU 없으면 여기서 즉시 중단

In [ ]:
import sys, torch

print(f'Python: {sys.version_info[:3]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}, available: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        '\n\n[!] GPU 런타임 아님!\n'
        '    [상단 메뉴] 런타임 → 런타임 유형 변경 → T4 GPU 선택 → 저장\n'
        '    그 후 노트북 다시 모두 실행.\n'
    )

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('\n환경 OK.')

## 2. Google Drive 마운트 (체크포인트 영구 저장)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

DRIVE_CKPT = '/content/drive/MyDrive/medical_ai_ckpts'
os.makedirs(DRIVE_CKPT, exist_ok=True)

existing = sorted(os.listdir(DRIVE_CKPT))
print(f'Drive 체크포인트 폴더: {DRIVE_CKPT}')
print(f'기존 파일 ({len(existing)}개):')
for f in existing:
    size = os.path.getsize(os.path.join(DRIVE_CKPT, f)) // 1024
    print(f'  {f}: {size} KB')

## 3. 안전 클론 (코드만 받기, checkpoints는 절대 건드리지 않음)

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/codingbear107/medical_ai.git'
PROJECT_DIR = '/content/medical_ai'
DRIVE_CKPT = '/content/drive/MyDrive/medical_ai_ckpts'

os.chdir('/content')

# 코드 받기 — 이미 있으면 git pull, 없으면 clone
if os.path.exists(PROJECT_DIR) and os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    print('기존 리포 발견 → git pull')
    r = subprocess.run(['git', '-C', PROJECT_DIR, 'pull'],
                       capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print('pull 실패, 강제 재클론')
        subprocess.run(['rm', '-rf', PROJECT_DIR])
        subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
else:
    print('신규 클론')
    if os.path.exists(PROJECT_DIR):
        subprocess.run(['rm', '-rf', PROJECT_DIR])
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

# checkpoints/ 가 일반 폴더로 존재하면 (학습 결과 있음) Drive로 백업
local_ckpt = os.path.join(PROJECT_DIR, 'checkpoints')
if os.path.isdir(local_ckpt) and not os.path.islink(local_ckpt):
    import shutil
    for f in os.listdir(local_ckpt):
        src = os.path.join(local_ckpt, f)
        dst = os.path.join(DRIVE_CKPT, f)
        if not os.path.exists(dst):  # Drive에 없는 것만 복사
            shutil.copy2(src, dst)
            print(f'  Drive로 백업: {f}')
    subprocess.run(['rm', '-rf', local_ckpt])

# symlink 보장 — checkpoints/ → DRIVE_CKPT
if not os.path.exists(local_ckpt):
    os.symlink(DRIVE_CKPT, local_ckpt)
    print(f'symlink 설정: {local_ckpt} → {DRIVE_CKPT}')
elif os.path.islink(local_ckpt):
    print(f'symlink 이미 존재: {local_ckpt} → {os.path.realpath(local_ckpt)}')

os.chdir(os.path.join(PROJECT_DIR, 'src'))
if os.path.join(PROJECT_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

print(f'\nPWD: {os.getcwd()}')
print(f'src 파일 수: {len(os.listdir("."))}')
print(f'checkpoints 안 파일: {sorted(os.listdir("../checkpoints"))}')

## 4. 패키지 설치 (medmnist만 추가)

In [ ]:
!pip install -q medmnist
import medmnist
print(f'medmnist {medmnist.__version__} OK')

## 5. 파이프라인 자동 실행 (약 50분)

axial부터 모두 새로 학습. runner.py 우회하고 `!python -u` 직접 호출 → 출력 실시간 보임.

- View A 학습 (10 epoch, ~18분)
- View C 학습 (10 epoch, ~18분)
- Merge grid search (~5분)
- Few-shot fine-tuning (20 epoch, ~12분)

In [ ]:
import os, time

# 0. 정리: axial부터 모두 새로 학습
ckpt = '/content/medical_ai/checkpoints'
for f in ['axial_model.pth', 'axial_fisher.pth', 'axial_params.pth',
          'coronal_model.pth', 'coronal_fisher.pth', 'coronal_params.pth',
          'base_model.pth', 'merge_grid_results.csv',
          'fewshot_model.pth', 'model.pth']:
    p = os.path.join(ckpt, f)
    if os.path.exists(p):
        os.remove(p)
print('정리 완료. 남은:', sorted(os.listdir(ckpt)))

t_total = time.time()

# 1. View A 학습 (10 epoch)
print('\n' + '=' * 70)
print('  [1/4] View A 학습 (10 epoch, 약 18분)')
print('=' * 70)
!python -u train_view.py --view A --dataset pathmnist --epochs 10 --seed 42 --save_init

# 2. View C 학습 (10 epoch)
print('\n' + '=' * 70)
print('  [2/4] View C 학습 (10 epoch, 약 18분)')
print('=' * 70)
!python -u train_view.py --view C --dataset pathmnist --epochs 10 --seed 42

# 3. Merge grid search
print('\n' + '=' * 70)
print('  [3/4] Merge grid search (약 5분)')
print('=' * 70)
!python -u merge_and_eval.py --dataset pathmnist --seed 42

# 4. Few-shot fine-tuning
print('\n' + '=' * 70)
print('  [4/4] Few-shot fine-tuning (20 epoch, 약 12분)')
print('=' * 70)
!python -u train_fewshot.py --dataset pathmnist --seed 42 --epochs 20

print(f'\n{"=" * 70}')
print(f'  파이프라인 완료. 총 {(time.time() - t_total) / 60:.1f}분')
print(f'{"=" * 70}')
print(f'  체크포인트:')
for f in sorted(os.listdir(ckpt)):
    print(f'     {f}: {os.path.getsize(os.path.join(ckpt, f)) // 1024} KB')

## 6. (선택) Merge grid 결과 자세히 보기

In [ ]:
import os, pandas as pd
csv_path = '/content/medical_ai/checkpoints/merge_grid_results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(df.sort_values('sag_f1', ascending=False).to_string(index=False))
else:
    print('merge_grid_results.csv 아직 없음')

## 7. 본 게임 데이터 도착 후 (Stage 2)

위 셀들은 그대로 두고 다음 절차:
1. Drive에 `data/axial/`, `data/coronal/`, `data/sagittal/` 폴더 업로드
2. 셀 5의 `--dataset pathmnist`를 `--dataset real`로 모두 변경
3. epoch도 늘리기 (예: `--epochs 50` for view, `--epochs 50` for fewshot)
4. 모두 실행

In [ ]:
# 본 게임 실행 예시 (지금은 실행하지 마세요)
# import os
# ckpt = '/content/medical_ai/checkpoints'
# # 모든 PathMNIST 결과 정리
# for f in os.listdir(ckpt):
#     os.remove(os.path.join(ckpt, f))
#
# !python -u train_view.py --view A --dataset real --epochs 50 --seed 42 --save_init
# !python -u train_view.py --view C --dataset real --epochs 50 --seed 42
# !python -u merge_and_eval.py --dataset real --seed 42
# !python -u train_fewshot.py --dataset real --seed 42 --epochs 50